# 02. Image Preprocessing, Denoising & Co-Registration
### Project: Image Change Detection using Digital Image Processing for Environmental Monitoring

This notebook demonstrates:
1. Spatial noise reduction using **Gaussian Blur** vs **Median Filtering**.
2. **Radiometric Normalization** via Histogram Matching to eliminate seasonal illumination and atmospheric discrepancies.
3. Automated **Cloud and Bright Haze Masking**.
4. Multi-temporal image **Co-Registration**:
   - Georeferenced scenes: `rasterio.warp.reproject`
   - Unreferenced RGB scenes: OpenCV **ORB** feature detection + **RANSAC** homography alignment.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent / "backend"))

from app.services.utils import load_image, generate_synthetic_pair, normalize_to_uint8
from app.services.preprocessing import denoise, histogram_match, cloud_mask, preprocess_pair
from app.services.registration import align_pair


### Step 1: Load Image Pair


In [ ]:
project_root = Path.cwd().parent
pair_paths = generate_synthetic_pair("deforestation", size=512, base_dir=str(project_root))

before_info = load_image(pair_paths["before_path"])
after_info = load_image(pair_paths["after_path"])

print(f"Loaded Before Shape: {before_info['shape']}")
print(f"Loaded After Shape:  {after_info['shape']}")


### Step 2: Denoising Algorithms Comparison
We compare Gaussian smoothing ($k=5$) against Median filtering on noisy satellite channels.


In [ ]:
noisy_sample = before_info["bands"][2].copy()  # NIR band

# Denoising
gauss_denoised = denoise(noisy_sample, method="gaussian", ksize=5)
median_denoised = denoise(noisy_sample, method="median", ksize=5)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(noisy_sample, cmap="viridis")
axes[0].set_title("Original NIR Band", fontsize=12, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(gauss_denoised, cmap="viridis")
axes[1].set_title("Gaussian Filter (k=5)", fontsize=12, fontweight="bold")
axes[1].axis("off")

axes[2].imshow(median_denoised, cmap="viridis")
axes[2].set_title("Median Filter (k=5)", fontsize=12, fontweight="bold")
axes[2].axis("off")

plt.tight_layout()
plt.show()


### Step 3: Radiometric Calibration via Histogram Matching
Different acquisition dates have varying sun angles, atmospheric haze, and soil moisture. Histogram matching calibrates the target image's histogram to match the reference image.


In [ ]:
# Add synthetic atmospheric shift to after image
shifted_after = after_info["bands"] * 1.25 + 0.04
matched_after = histogram_match(shifted_after, before_info["bands"])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Before NIR
axes[0, 0].imshow(before_info["bands"][2], cmap="viridis")
axes[0, 0].set_title("Reference (Before NIR)", fontweight="bold")
axes[0, 0].axis("off")

# Uncalibrated After NIR
axes[0, 1].imshow(shifted_after[2], cmap="viridis")
axes[0, 1].set_title("Uncalibrated (After NIR)", fontweight="bold")
axes[0, 1].axis("off")

# Histogram before matching
axes[1, 0].hist(before_info["bands"][2].ravel(), bins=50, color="blue", alpha=0.6, label="Reference")
axes[1, 0].hist(shifted_after[2].ravel(), bins=50, color="orange", alpha=0.5, label="Uncalibrated")
axes[1, 0].set_title("Before Histogram Matching", fontweight="bold")
axes[1, 0].legend()
axes[1, 0].grid(True, linestyle="--", alpha=0.5)

# Histogram after matching
axes[1, 1].hist(before_info["bands"][2].ravel(), bins=50, color="blue", alpha=0.6, label="Reference")
axes[1, 1].hist(matched_after[2].ravel(), bins=50, color="green", alpha=0.5, label="Matched After")
axes[1, 1].set_title("After Histogram Matching (Radiometrically Calibrated)", fontweight="bold")
axes[1, 1].legend()
axes[1, 1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()


### Step 4: Spatial Co-Registration & Alignment


In [ ]:
aligned_after, reg_msg = align_pair(before_info, after_info)
print(f"Registration Output: {reg_msg}")
print(f"Aligned Array Shape: {aligned_after.shape}")


### Step 5: Full Automated Preprocessing Pipeline


In [ ]:
clean_before, clean_after, valid_mask = preprocess_pair(
    before_info["bands"],
    aligned_after,
    do_denoise=True,
    do_match=True
)

print(f"Cleaned Before Shape: {clean_before.shape}")
print(f"Cleaned After Shape:  {clean_after.shape}")
print(f"Valid Pixels Percentage: {valid_mask.mean() * 100:.2f}%")
